# 🎮 什麼時候 GPU 才真的比較快？— cuDF Game Analytics Benchmark

用 **RAPIDS `cudf.pandas`** 誠實實測 GPU 加速的邊界。

**兩個發現:**
1. **天真移植的 pandas 分析(RFM/留存/漏斗)GPU 不一定贏** —— 因為 `qcut`/`rank`/`np.select`/Python `set` 會 fallback 回 CPU。
2. **改寫成 GPU 原生運算(groupby/merge/sort)+ 放大資料量後,GPU 才大贏。**

**How to run:** Runtime → T4 GPU。`USE_GPU=False` 跑一趟 → Restart → `USE_GPU=True` 再跑一趟 → 第 6 節填數字自動畫圖。

In [ ]:
USE_GPU = True  # 改 False 跑 pandas baseline（改完必須 Runtime→Restart session 才生效）

if USE_GPU:
    %load_ext cudf.pandas

In [ ]:
import pandas as pd
import numpy as np
import time

print("backend check:", pd)

## 1. 產生合成遊戲事件資料（20M rows）

In [ ]:
N_PLAYERS = 500_000
N_EVENTS = 20_000_000
rng = np.random.default_rng(42)

t0 = time.perf_counter()
player_ids = rng.integers(0, N_PLAYERS, N_EVENTS)
days = rng.exponential(scale=25, size=N_EVENTS).astype(int) % 90
events = pd.DataFrame({
    "player_id": player_ids,
    "day": days,
    "event_type": rng.choice(
        ["login", "stage_clear", "gacha", "purchase"],
        size=N_EVENTS, p=[0.45, 0.35, 0.15, 0.05]),
    "revenue": np.where(
        rng.random(N_EVENTS) < 0.05,
        rng.lognormal(mean=3.0, sigma=1.0, size=N_EVENTS), 0.0),
})
events["install_day"] = events.groupby("player_id")["day"].transform("min")
print(f"generated {len(events):,} rows in {time.perf_counter()-t0:.1f}s")
events.head()

## 2. RFM 分群 — ⚠️ 含 `qcut`/`rank`/`np.select`（GPU 會 fallback）

In [ ]:
t0 = time.perf_counter()
rfm = events.groupby("player_id").agg(
    recency=("day", "max"),
    frequency=("event_type", "count"),
    monetary=("revenue", "sum"),
)
rfm["recency"] = 89 - rfm["recency"]
rfm["R"] = pd.qcut(rfm["recency"], 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["segment"] = np.select(
    [
        (rfm.R >= 3) & (rfm.F >= 3) & (rfm.M >= 3),
        (rfm.R >= 3) & (rfm.F >= 3),
        (rfm.R <= 2) & (rfm.M >= 3),
        (rfm.R <= 2) & (rfm.F <= 2),
    ],
    ["whale_active", "core_engaged", "churning_payer", "at_risk"],
    default="casual",
)
t_rfm = time.perf_counter() - t0
print(f"RFM done in {t_rfm:.2f}s")
rfm["segment"].value_counts()

## 3. 留存 Cohort（D1 / D7 / D30）— 群組數少，GPU 開銷難攤平

In [ ]:
t0 = time.perf_counter()
logins = events[events.event_type == "login"][["player_id", "day", "install_day"]].copy()
logins["age"] = logins["day"] - logins["install_day"]
cohort_size = logins.groupby("install_day")["player_id"].nunique()
retention = {}
for d in [1, 7, 30]:
    ret = logins[logins.age == d].groupby("install_day")["player_id"].nunique()
    retention[f"D{d}"] = (ret / cohort_size).fillna(0)
retention_df = pd.DataFrame(retention)
t_ret = time.perf_counter() - t0
print(f"retention done in {t_ret:.2f}s")
retention_df.head(10)

## 4. 付費漏斗 — ❌ 版本 A：Python `set` 交集（GPU 完全使不上力）

In [ ]:
t0 = time.perf_counter()
funnel_sets = {
    step: set(events.loc[events.event_type == step, "player_id"].unique())
    for step in ["login", "stage_clear", "gacha", "purchase"]
}
base = funnel_sets["login"]
funnel = {}
current = base
for step in ["login", "stage_clear", "gacha", "purchase"]:
    current = current & funnel_sets[step]
    funnel[step] = len(current) / max(len(base), 1)
t_funnel = time.perf_counter() - t0
print(f"funnel (Python set) done in {t_funnel:.2f}s")
funnel

## 4b. 付費漏斗 — ✅ 版本 B：GPU 原生改寫（groupby-max 指示欄）

同一個商業問題,改用 cuDF **原生支援**的 groupby/布林聚合,避開 Python `set` 與 host 搬移。
這就是「把 fallback 運算改寫成 GPU 路徑」的示範 —— 觀察它在 GPU 上是否翻盤。

In [ ]:
t0 = time.perf_counter()
steps = ["login", "stage_clear", "gacha", "purchase"]
ev = events[["player_id", "event_type"]].copy()
for s in steps:
    ev[s] = (ev["event_type"] == s)
flags = ev.groupby("player_id")[steps].max()   # 每位玩家是否到達各步驟（原生 groupby-max）
funnel_native = {}
mask = flags["login"].astype("bool")
for s in steps:
    mask = mask & flags[s].astype("bool")
    funnel_native[s] = float(mask.mean())
t_funnel_native = time.perf_counter() - t0
print(f"funnel (GPU-native) done in {t_funnel_native:.2f}s")
funnel_native

## 5. 大資料 × GPU 原生運算（50M rows）— GPU 的主場

前面的分析要嘛資料不夠大、要嘛運算 fallback。這裡用 **cuDF 原生支援**且**計算密集**的三種運算
（大型排序、多鍵聚合、雜湊 join）在 50M 列上測試 —— 這才是 GPU 該大贏的場景。

In [ ]:
M = 50_000_000
t0 = time.perf_counter()
big = pd.DataFrame({
    "key": rng.integers(0, 1_000_000, M).astype("int32"),
    "val": (rng.random(M) * 100).astype("float32"),
    "qty": rng.integers(1, 100, M).astype("int32"),
})
dim = pd.DataFrame({
    "key": np.arange(1_000_000, dtype="int32"),
    "tier": rng.integers(0, 5, 1_000_000).astype("int32"),
})
print(f"generated {len(big):,} rows in {time.perf_counter()-t0:.1f}s")

t0 = time.perf_counter()
_ = big.sort_values("val")
t_sort = time.perf_counter() - t0
print(f"sort_values (50M):        {t_sort:.2f}s")

t0 = time.perf_counter()
_ = big.groupby("key").agg(
    total=("val", "sum"), avg=("val", "mean"),
    cnt=("qty", "count"), mx=("val", "max"), mn=("val", "min"),
)
t_grp = time.perf_counter() - t0
print(f"groupby multi-agg (1M grp): {t_grp:.2f}s")

t0 = time.perf_counter()
_ = big.merge(dim, on="key", how="left")
t_merge = time.perf_counter() - t0
print(f"merge/join (50M x 1M):     {t_merge:.2f}s")

## 6. 結果彙整 + 加速比視覺化

跑完 **CPU（USE_GPU=False）** 與 **GPU（USE_GPU=True）** 兩趟後,
把兩趟下方 print 出的數字分別填入 `cpu` / `gpu` dict,執行即產生對比圖。

In [ ]:
labels = ["RFM", "Retention", "Funnel(set)", "Funnel(native)", "Sort50M", "Groupby50M", "Merge50M"]
current = [t_rfm, t_ret, t_funnel, t_funnel_native, t_sort, t_grp, t_merge]
print("backend:", "cudf.pandas (GPU)" if USE_GPU else "pandas (CPU)")
for l, v in zip(labels, current):
    print(f"  {l:16s} {v:.2f}s")

In [ ]:
import matplotlib.pyplot as plt

# 把兩趟實跑數字填入（順序同上：RFM, Retention, Funnel_set, Funnel_native, Sort, Groupby, Merge）
cpu = {"RFM": 0.39, "Retention": 0.42, "Funnel(set)": 4.11,
       "Funnel(native)": None, "Sort50M": None, "Groupby50M": None, "Merge50M": None}
gpu = {"RFM": 0.65, "Retention": 0.69, "Funnel(set)": 3.63,
       "Funnel(native)": None, "Sort50M": None, "Groupby50M": None, "Merge50M": None}

steps = [k for k in cpu if cpu[k] is not None and gpu[k] is not None]
speedup = [cpu[s] / gpu[s] for s in steps]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = range(len(steps)); w = 0.38
axes[0].bar([i - w/2 for i in x], [cpu[s] for s in steps], w, label="pandas (CPU)", color="#4b5563")
axes[0].bar([i + w/2 for i in x], [gpu[s] for s in steps], w, label="cudf.pandas (GPU)", color="#76b900")
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(steps, rotation=30, ha="right")
axes[0].set_ylabel("seconds"); axes[0].set_title("Execution time: CPU vs GPU")
axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)

colors = ["#76b900" if s >= 1 else "#dc2626" for s in speedup]
bars = axes[1].bar(steps, speedup, color=colors)
axes[1].axhline(1, color="#111827", ls="--", lw=1, label="1× (break-even)")
axes[1].set_ylabel("speedup (×)"); axes[1].set_title("GPU speedup (>1 = GPU wins, red = GPU loses)")
axes[1].set_xticklabels(steps, rotation=30, ha="right")
axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)
for b, s in zip(bars, speedup):
    axes[1].text(b.get_x() + b.get_width()/2, s, f"{s:.1f}×", ha="center", va="bottom")

plt.tight_layout()
plt.savefig("benchmark_result.png", dpi=120, bbox_inches="tight")
plt.show()
print("saved benchmark_result.png")

## PM Takeaways

- **GPU 不是萬靈丹**:天真移植的 RFM/留存/漏斗,因為 `qcut`/`rank`/`np.select`/Python `set` 觸發 CPU fallback,GPU 反而更慢。
- **判斷力才值錢**:導入 GPU 前要問兩件事 ——(1) 資料量夠大攤平開銷嗎?(2) 運算走原生 GPU 路徑還是 fallback?
- **改寫可以翻盤**:把漏斗從 Python `set` 改寫成原生 groupby(第 4b 節),以及在大資料的 sort/merge/groupby(第 5 節),GPU 才展現真正實力。
- **`cudf.pandas` 的價值**:零改碼讓「先試試看 GPU 划不划算」的成本趨近於零,不吃 GPU 的運算會自動 fallback、不會壞掉 —— 對數據團隊是低風險的加速實驗管道。

*Author: Jerome Kuo — gaming data PM, NVIDIA DLI certified (深度學習基礎理論與實踐).*